In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
!pip install seqeval
from seqeval.metrics import classification_report, f1_score

raw_data = [
    (
        ["Barack", "Obama", "was", "born", "in", "Hawaii", "."],
        ["B-PER", "I-PER", "O", "O", "O", "B-LOC", "O"]
    ),
    (
        ["Apple", "unveiled", "new", "products", "in", "California", "."],
        ["B-ORG", "O", "O", "O", "O", "B-LOC", "O"]
    ),
    (
        ["Alice", "visited", "Paris", "with", "Bob", "Smith", "."],
        ["B-PER", "O", "B-LOC", "O", "B-PER", "I-PER", "O"]
    ),
    (
        ["Google", "headquarters", "are", "located", "in", "Mountain", "View", "."],
        ["B-ORG", "O", "O", "O", "O", "B-LOC", "I-LOC", "O"]
    )
]

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
IGNORE_INDEX = -100

words = sorted(list({w for sent, _ in raw_data for w in sent}))
word2idx = {w: i + 2 for i, w in enumerate(words)}
word2idx[PAD_TOKEN] = 0
word2idx[UNK_TOKEN] = 1

tags = sorted(list({t for _, tag_seq in raw_data for t in tag_seq}))
tag2idx = {t: i for i, t in enumerate(tags)}
idx2tag = {i: t for t, i in tag2idx.items()}

class NERDataset(Dataset):
    def __init__(self, data, word2idx, tag2idx):
        self.samples = []
        for words, tags in data:
            x = [word2idx.get(w, word2idx[UNK_TOKEN]) for w in words]
            y = [tag2idx[t] for t in tags]
            self.samples.append((torch.tensor(x, dtype=torch.long),
                                 torch.tensor(y, dtype=torch.long)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def pad_collate_fn(batch):
    sequences, labels = zip(*batch)
    padded_seqs = nn.utils.rnn.pad_sequence(sequences, batch_first=True, padding_value=word2idx[PAD_TOKEN])
    padded_labels = nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=IGNORE_INDEX)
    return padded_seqs, padded_labels

train_dataset = NERDataset(raw_data, word2idx, tag2idx)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=pad_collate_fn)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16250 sha256=038bf304561c84d8301383573821f1c137c4c794f4b72c4efc92847b411aa233
  Stored in directory: /root/.cache/pip/wheels/14/cf/a7/8f28ef376d707ff10e3922899482a2f23ef3002f8a952f47ac
Successfully built seqeval


In [ ]:
class BiLSTMTokenClassifier(nn.Module):
    def __init__(self, vocab_size, num_tags, embed_dim=64, hidden_dim=128, num_layers=1, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True
        )
        self.classifier = nn.Linear(hidden_dim * 2, num_tags)

    def forward(self, x):
        embeds = self.dropout(self.embedding(x))
        lstm_out, _ = self.lstm(embeds)
        logits = self.classifier(self.dropout(lstm_out))
        return logits

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BiLSTMTokenClassifier(
    vocab_size=len(word2idx),
    num_tags=len(tag2idx),
    embed_dim=32,
    hidden_dim=64
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)

epochs = 30
model.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits.view(-1, len(tag2idx)), batch_y.view(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        epoch_loss += loss.item()

model.eval()
true_tags_all = []
pred_tags_all = []

eval_loader = DataLoader(train_dataset, batch_size=1, shuffle=False)

with torch.no_grad():
    for x_eval, y_eval in eval_loader:
        x_eval = x_eval.to(device)
        logits = model(x_eval)
        predictions = torch.argmax(logits, dim=-1)[0]
        targets = y_eval[0]

        sent_preds = [idx2tag[p.item()] for p in predictions]
        sent_trues = [idx2tag[t.item()] for t in targets]

        pred_tags_all.append(sent_preds)
        true_tags_all.append(sent_trues)

print("\n--- NER Sequence Evaluation Report ---")
print(classification_report(true_tags_all, pred_tags_all))
print(f"Overall Micro F1 Score: {f1_score(true_tags_all, pred_tags_all):.4f}")


--- NER Sequence Evaluation Report ---
              precision    recall  f1-score   support

         LOC       1.00      1.00      1.00         4
         ORG       1.00      1.00      1.00         2
         PER       1.00      1.00      1.00         3

   micro avg       1.00      1.00      1.00         9
   macro avg       1.00      1.00      1.00         9
weighted avg       1.00      1.00      1.00         9

Overall Micro F1 Score: 1.0000
